# Module 15 lab · Looking inside: interpretability and safety

The code from every lesson of [this module](https://aicourses.dev/research/interpretability-and-safety), in order. Read a lesson, then run its cells here: each lesson builds on the one before, so run the cells from the top the first time.

**First time?** Read the [one-time setup](https://aicourses.dev/research/labs): a Google account, your own copy, and what to expect.

The lessons that explain this code are part of [AI Courses Pro](https://aicourses.dev/#pricing); the module’s first lesson is free.

Under each cell, *The lesson shows* is what it should print. Your last digits can differ a little.

**Keep your work:** File → Save a copy in Drive. Colab does not keep changes to this copy.

## 15.2 The residual stream and the logit lens

The residual from Module 10 is a running total that every layer reads from and adds to. The logit lens reads the model’s best guess from that total after each layer, so you watch the answer form, layer by layer.

[Read the lesson](https://aicourses.dev/research/interpretability-and-safety/residual-stream-and-logit-lens)

### The logit lens in code

The toy example first, in PyTorch:

In [ ]:
import torch

words = ["the", "Rome", "Paris"]
U = torch.tensor([[1., 0.], [0., 1.], [-2., 2.]])  # one row per word
x = torch.tensor([2., 0.])               # the stream at the start
writes = [torch.tensor([-1., 1.5]),      # what layer 1 adds
          torch.tensor([-1., 0.5])]      # what layer 2 adds

for layer in range(3):
    scores = U @ x                       # a dot product per word
    print(layer, x.tolist(), scores.tolist(), words[scores.argmax()])
    if layer < 2:
        x = x + writes[layer]            # add, never overwrite
# 0 [2.0, 0.0] [2.0, 0.0, -4.0] the
# 1 [1.0, 1.5] [1.0, 1.5, 1.0] Rome
# 2 [0.0, 2.0] [0.0, 2.0, 4.0] Paris

Now GPT-2 small, with Hugging Face Transformers. Asking for `output_hidden_states=True` returns the stream after every block:

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").eval()
ids = tok("The Eiffel Tower is in the city of", return_tensors="pt")
with torch.no_grad():
    out = model(**ids, output_hidden_states=True)

# 13 snapshots of the stream: before block 1, then after each block.
# The last snapshot has already been through the final layer norm.
with torch.no_grad():
    for layer, h in enumerate(out.hidden_states[:-1]):
        x = h[0, -1]                     # the stream at the last token
        logits = model.lm_head(model.transformer.ln_f(x))
        print(layer, tok.decode(logits.argmax()))
print(12, tok.decode(out.logits[0, -1].argmax()))

## 15.3 Induction heads: a mechanism in your own GPT

An induction head is an attention head that, helped by a head in an earlier layer, spots a repeat and copies what came after it last time. It is one of the clearest known cases of one part of a model doing one job, and you test your own GPT for it.

[Read the lesson](https://aicourses.dev/research/interpretability-and-safety/induction-heads)

### Find them in GPT-2 small

Asking Hugging Face for attention weights needs the plain, “eager” attention code, because the faster versions do not return them:

In [ ]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "gpt2", attn_implementation="eager").eval()  # eager keeps weights

torch.manual_seed(0)
T = 50
half = torch.randint(0, 50257, (4, T))    # 4 runs of random tokens
ids = torch.cat([half, half], dim=1)      # each one repeated: 100 long
with torch.no_grad():
    out = model(ids, output_attentions=True)

scores = torch.zeros(12, 12)              # one score per layer and head
for layer, att in enumerate(out.attentions):  # [4, 12, 100, 100]
    # weight from each token in copy 2 to the token after its twin
    stripe = att.diagonal(offset=1 - T, dim1=2, dim2=3)[:, :, 1:]
    scores[layer] = stripe.mean(dim=(0, 2))

best = scores.flatten().topk(5)
for s, i in zip(best.values, best.indices):
    print(f"head {i // 12}.{i % 12}: {s:.2f}")

### Test your own GPT

Your GPT from Module 10 computes attention weights by hand, so you can save them. In your attention module’s forward pass, right after the softmax, add `self.att = att.detach()`. Then run the same test, with random tokens drawn from your own vocabulary:

In [ ]:
def induction_scores(model, vocab_size, T=50, n=4):
    torch.manual_seed(0)
    half = torch.randint(0, vocab_size, (n, T))
    ids = torch.cat([half, half], dim=1)     # needs context of 2T
    with torch.no_grad():
        model(ids)
    rows = []
    for block in model.blocks:               # rename to match your code
        att = block.attn.att                 # [n, heads, 2T, 2T]
        stripe = att.diagonal(offset=1 - T, dim1=2, dim2=3)[:, :, 1:]
        rows.append(stripe.mean(dim=(0, 2)))
    return torch.stack(rows)                 # [layers, heads]

## 15.4 Probes and steering vectors

A probe is a small model that checks whether an idea, such as “this is a refusal”, is written in a model’s numbers. A steering vector adds or removes that idea while the model runs. Both are cheap, and labs use them a lot.

[Read the lesson](https://aicourses.dev/research/interpretability-and-safety/probes-and-steering)

### Train a probe on GPT-2 small

Build 96 short reviews from a template, “The film was wonderful.”, with six things and sixteen adjectives, and keep the stream at the full stop after every block:

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").eval()

things = ["film", "meal", "hotel", "book", "concert", "holiday"]
good = ["wonderful", "great", "lovely", "excellent",
        "brilliant", "fantastic", "superb", "delightful"]
bad = ["terrible", "awful", "boring", "dreadful",
       "horrible", "disappointing", "miserable", "poor"]

def streams(words):      # the last token's stream, after every block
    rows = []
    for t in things:
        for w in words:
            ids = tok(f"The {t} was {w}.", return_tensors="pt")
            with torch.no_grad():
                out = model(**ids, output_hidden_states=True)
            hs = out.hidden_states
            rows.append(torch.stack([h[0, -1] for h in hs]))
    return torch.stack(rows)          # [sentences, 13 snapshots, 768]

Train on six good and six bad adjectives and test on the last two of each, words the probe never saw:

In [ ]:
Xtr = torch.cat([streams(good[:6]), streams(bad[:6])])  # 72 sentences
Xte = torch.cat([streams(good[6:]), streams(bad[6:])])  # 24, new words
ytr = torch.tensor([1.] * 36 + [0.] * 36)               # 1 = positive
yte = torch.tensor([1.] * 12 + [0.] * 12)

def probe_accuracy(layer):
    torch.manual_seed(0)
    X, Z = Xtr[:, layer], Xte[:, layer]
    mu, sd = X.mean(0), X.std(0) + 1e-6     # rescale each number
    probe = torch.nn.Linear(768, 1)         # 768 weights and a bias
    opt = torch.optim.Adam(probe.parameters(), lr=1e-2)
    for _ in range(300):
        logit = probe((X - mu) / sd).squeeze(1)
        loss = F.binary_cross_entropy_with_logits(logit, ytr)
        opt.zero_grad(); loss.backward(); opt.step()
    guess = probe((Z - mu) / sd).squeeze(1) > 0
    return (guess == yte.bool()).float().mean().item()

print([round(probe_accuracy(layer), 2) for layer in range(13)])
# [0.5, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]

### From a direction to a dial

Now steer GPT-2 small, adding $\alpha \mathbf{d}$ to the stream after 6 blocks, at every position, while it writes:

In [ ]:
L = 6                    # steer the stream after 6 blocks
d = Xtr[:36, L].mean(0) - Xtr[36:, L].mean(0)  # positive - negative

def generate(prompt, alpha):
    def add(module, args):               # runs just before block L
        return (args[0] + alpha * d,) + args[1:]
    handle = model.transformer.h[L].register_forward_pre_hook(add)
    ids = tok(prompt, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=12,
                             do_sample=False, pad_token_id=50256)
    handle.remove()
    return tok.decode(out[0, ids.input_ids.shape[1]:])

for alpha in [-2, 0, 2]:
    print(alpha, repr(generate("My week has been", alpha)))

## 15.5 Activation patching: testing what causes what

A probe shows that a model’s numbers hold an idea, not that the model uses it. Activation patching copies numbers from one run into another and watches whether the answer changes. That shows what actually causes what.

[Read the lesson](https://aicourses.dev/research/interpretability-and-safety/activation-patching)

### Patching in code

First run the clean prompt, keep its residual stream, and measure both runs:

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").eval()
clean = tok("The Eiffel Tower is in the city of",
            return_tensors="pt").input_ids
corrupt = tok("The Colosseum is in the city of",
              return_tensors="pt").input_ids    # also 10 tokens
paris, rome = tok(" Paris").input_ids[0], tok(" Rome").input_ids[0]

def diff(logits):                  # how far Paris is ahead of Rome
    return (logits[0, -1, paris] - logits[0, -1, rome]).item()

with torch.no_grad():
    out = model(clean, output_hidden_states=True)
    saved = out.hidden_states      # the clean stream, 13 snapshots
    d_clean, d_bad = diff(out.logits), diff(model(corrupt).logits)
print(f"clean {d_clean:.2f}, corrupted {d_bad:.2f}")
# clean 2.42, corrupted -1.49

Then rerun the corrupted prompt with a hook that pastes the clean numbers into chosen positions just before one block:

In [ ]:
def patch(layer, positions):
    def paste(module, args):       # runs just before block `layer`
        h = args[0].clone()
        h[0, positions] = saved[layer][0, positions]
        return (h,) + args[1:]
    block = model.transformer.h[layer]
    handle = block.register_forward_pre_hook(paste)
    with torch.no_grad():
        d = diff(model(corrupt).logits)
    handle.remove()
    return (d - d_bad) / (d_clean - d_bad)  # 0 = no change, 1 = fixed

subject, last = [1, 2, 3, 4], [9]
for layer in range(12):
    print(layer, round(patch(layer, subject), 2),
          round(patch(layer, last), 2))

## 15.6 Safety for engineers: from reward hacking to misalignment

A model is misaligned when it pursues goals other than the ones intended. RL runs can teach this by accident: cheating on rewards, behaving differently when the model senses a test, or hiding what it can do. You learn to spot each one in ordinary runs.

[Read the lesson](https://aicourses.dev/research/interpretability-and-safety/safety-for-engineers)

### Checks for your own runs

A few lines of code catch the commonest tells. This sketch flags a run whose reward still climbs while the held-out score falls, and scans a transcript for traces of gaming:

In [ ]:
import re

TELLS = {                      # traces that a model gamed its checks
    "exits early": r"sys\.exit\(0\)",
    "edits tests": r"(open|write).*tests?/",
    "skips tests": r"pytest\.skip|@unittest\.skip",
    "hard-codes": r"if .*== .*:\s*return ['\"\d]",
}

def tells(transcript):
    return [n for n, p in TELLS.items() if re.search(p, transcript)]

def diverging(reward, held_out, window=3):
    # the reward still climbs while a score it cannot game falls
    return (reward[-1] > reward[-1 - window]
            and held_out[-1] < held_out[-1 - window])

reward   = [0.20, 0.35, 0.50, 0.70, 0.85, 0.95]   # made-up run
held_out = [0.18, 0.30, 0.38, 0.36, 0.31, 0.27]
sample = "fix: if n == 7: return 13\nimport sys; sys.exit(0)"
print(diverging(reward, held_out), tells(sample))
# True ['exits early', 'hard-codes']